# Q-Table Continuous Analysis — MountainCar-v0

**Assignment RLI 22.00 — Tinder for RL**  
**Part 02 | Scenario 2: Continuous MountainCar, Cost-Sensitive Reward**

This notebook analyzes the trained Q-table continuous agent with cost penalty. It covers:
1. Learning curve (reward, cost, action counts over training)
2. Policy visualization — what action does the agent take at every (position, velocity) state?
3. Value surface — how good does the agent think each state is?
4. Visitation frequency — which states did the agent explore?
5. Phase portrait — trajectories of the trained agent in position-velocity space
6. Physical interpretation — connecting the learned policy to the mechanics of the problem

## 0. Imports and Setup

## 1. Load Trained Model and Metrics

In [ ]:
# Load the trained Q-table
q_table_path = os.path.join(CURRENT_DIR, "results", "models", "continuous_q_table.npy")
q_table = np.load(q_table_path)

# Load metrics
rewards = np.load(os.path.join(CURRENT_DIR, "results", "metrics", "continuous_rewards.npy"))
action_counts = np.load(os.path.join(CURRENT_DIR, "results", "metrics", "continuous_action_counts.npy"))
costs = np.load(os.path.join(CURRENT_DIR, "results", "metrics", "continuous_costs.npy"))

print(f"Q-table shape: {q_table.shape}")
print(f"Total episodes: {len(rewards)}")
print(f"Max reward: {np.max(rewards):.2f}")
print(f"Min reward: {np.min(rewards):.2f}")
print(f"Average action count: {np.mean(action_counts):.2f}")
print(f"Average cost: {np.mean(costs):.2f}")

## 2. Learning Curves

In [ ]:
# Plot learning curves
fig, axes = plt.subplots(3, 1, figsize=(12, 12))

# Rewards
axes[0].plot(rewards, alpha=0.7)
axes[0].set_title("Episode Rewards")
axes[0].set_xlabel("Episode")
axes[0].set_ylabel("Total Reward")
axes[0].grid(True)

# Action counts
axes[1].plot(action_counts, alpha=0.7, color='orange')
axes[1].set_title("Non-Null Actions per Episode")
axes[1].set_xlabel("Episode")
axes[1].set_ylabel("Action Count")
axes[1].grid(True)

# Costs
axes[2].plot(costs, alpha=0.7, color='red')
axes[2].set_title("Total Cost per Episode")
axes[2].set_xlabel("Episode")
axes[2].set_ylabel("Cost")
axes[2].grid(True)

plt.tight_layout()
plt.savefig(os.path.join(CURRENT_DIR, "results", "learning_curves_continuous.png"), dpi=150, bbox_inches='tight')
plt.show()

## 3. Policy Visualization

In [ ]:
# Recreate agent for discretization
env = gym.make("MountainCar-v0")
agent = QTableAgent(
    state_low=env.observation_space.low,
    state_high=env.observation_space.high,
    num_bins=[100, 100],
    num_actions=3,
)
agent.q_table = q_table

# Create policy grid
pos_bins = np.linspace(env.observation_space.low[0], env.observation_space.high[0], 100)
vel_bins = np.linspace(env.observation_space.low[1], env.observation_space.high[1], 100)

policy = np.zeros((100, 100))
for i, pos in enumerate(pos_bins):
    for j, vel in enumerate(vel_bins):
        state = agent.discretize_state([pos, vel])
        policy[i, j] = np.argmax(q_table[state])

# Plot policy
fig, ax = plt.subplots(figsize=(10, 8))
cmap = plt.cm.get_cmap('viridis', 3)
im = ax.imshow(policy.T, origin='lower', extent=[pos_bins[0], pos_bins[-1], vel_bins[0], vel_bins[-1]], cmap=cmap, aspect='auto')
ax.set_title("Learned Policy")
ax.set_xlabel("Position")
ax.set_ylabel("Velocity")

# Colorbar
cbar = plt.colorbar(im, ax=ax, ticks=[0, 1, 2])
cbar.set_label("Action")
cbar.set_ticklabels(["Left", "Neutral", "Right"])

plt.savefig(os.path.join(CURRENT_DIR, "results", "policy_visualization_continuous.png"), dpi=150, bbox_inches='tight')
plt.show()

## 4. Value Surface

In [ ]:
# Value surface
value_surface = np.max(q_table, axis=2)

fig = plt.figure(figsize=(12, 8))
ax = fig.add_subplot(111, projection='3d')
X, Y = np.meshgrid(pos_bins, vel_bins)
ax.plot_surface(X, Y, value_surface.T, cmap='viridis')
ax.set_title("Value Surface")
ax.set_xlabel("Position")
ax.set_ylabel("Velocity")
ax.set_zlabel("Max Q-Value")

plt.savefig(os.path.join(CURRENT_DIR, "results", "value_surface_continuous.png"), dpi=150, bbox_inches='tight')
plt.show()

## 5. Visitation Frequency (Placeholder)

In [ ]:
# Placeholder for visitation frequency
# Would need to track during training
print("Visitation frequency not implemented yet.")

## 6. Phase Portrait (Placeholder)

In [ ]:
# Placeholder for phase portrait
print("Phase portrait not implemented yet.")

## 7. Physical Interpretation

In [ ]:
print("The continuous agent learns to minimize both time and action costs.")
print(f"Average actions per episode: {np.mean(action_counts):.2f}")
print(f"Average cost per episode: {np.mean(costs):.2f}")